# Assignment 1

INTC (Intel), 21 June 2012, LOBSTER level 10, 10:00–12:00

In [1]:
#Imports
from pathlib import Path

import numpy as np
import pandas as pd
from unito26.lob import lobster
from unito26.lob.messages import TickGrid

PATH = Path("../data/lobster/INTC_2012-06-21_34200000_57600000_message_10.csv")
WINDOW = lobster.TradingWindow (36000.0, 43200.0) #10:00 to 12:00
GRID = TickGrid(0.01)                             
UNIT = lobster.price_unit(GRID)                   #LOBSTER units per tick
DEPTH = 10

print("data found:", PATH.exists(), "LOBSTER units per tick:", UNIT)

data found: True LOBSTER units per tick: 100


## i: Rebuilding the book and comparing it with LOBSTER.

In [2]:
files = lobster.LobsterFiles.parse(PATH)
messages, lobster_book = lobster.load_aligned(files, WINDOW)

print ("messages:", messages.shape, "| book rows:", lobster_book.shape)
messages.head()

messages: (181245, 7) | book rows: (181245, 40)


,Time,TimeNanoseconds,Type,OrderID,Size,Price,Direction
0,36000.031697,36000031697218,1,46529037,100,273400,1
1,36000.031784,36000031783998,3,46510659,1400,273500,-1
2,36000.032643,36000032643004,1,46529267,1500,273400,1
3,36000.033928,36000033927721,3,46315764,100,273200,1
4,36000.045602,36000045601567,1,46531840,200,272700,1


In [3]:
# How many events of each LOBSTER type are in the window?
type_count = messages['Type'].value_counts()

In [4]:
from unito26.lob.assignment import to_message

# Check the translation on the first five messages of the window
for i in range(5):
    print(messages.iloc[i][["Type", "Size", "Price", "Direction"]].tolist(), "->", to_message(messages.iloc[i]))

[1.0, 100.0, 273400.0, 1.0] -> Message(time=36000.031697218, size=100, price=2734, direction=1, kind=<MessageType.SUBMIT: 1>)
[3.0, 1400.0, 273500.0, -1.0] -> Message(time=36000.031783998, size=1400, price=2735, direction=-1, kind=<MessageType.WITHDRAW: 2>)
[1.0, 1500.0, 273400.0, 1.0] -> Message(time=36000.032643004, size=1500, price=2734, direction=1, kind=<MessageType.SUBMIT: 1>)
[3.0, 100.0, 273200.0, 1.0] -> Message(time=36000.033927721, size=100, price=2732, direction=1, kind=<MessageType.WITHDRAW: 2>)
[1.0, 200.0, 272700.0, 1.0] -> Message(time=36000.045601567, size=200, price=2727, direction=1, kind=<MessageType.SUBMIT: 1>)


In [5]:
from unito26.lob.assignment import count_mismatches

result = count_mismatches(messages, lobster_book, UNIT, DEPTH)

print("Rows compared:", result["rows compared"])

print("Rows with mismatches:", result["rows with a mismatch"])

print("First mismatch in:", result["first mismatch"])

per_level = pd.Series(result["mismatches per level"], index=range(1, DEPTH+1))
per_level.name = "Rows"

print(per_level / result["rows compared"])

Rows compared: 181244
Rows with mismatches: 181181
First mismatch in: 64
1     0.415777
2     0.619811
3     0.823100
4     0.891395
5     0.950740
6     0.966664
7     0.977870
8     0.993754
9     0.995056
10    0.999652
Name: Rows, dtype: float64


In [6]:

from unito26.lob.orderbook import AggregateBook

first = result["first mismatch"]

print("message:", messages.iloc[first][["Time", "Type", "Size", "Price", "Direction"]].tolist())

# Fold up to that row again, and show level 10 in both books

our_book = AggregateBook.from_lobster_row(lobster_book.iloc[0], UNIT, DEPTH)

for i in range(1, first + 1):
    message = to_message(messages.iloc[i])
    if message is not None:
        our_book.apply(message, record=False)

level_10 = ["AskPrice10", "AskSize10", "BidPrice10", "BidSize10"]

pd.DataFrame(
    [our_book.to_lobster_row(UNIT, DEPTH)[36:40], lobster_book.iloc[first][level_10].tolist()],
    index=["our book", "LOBSTER"], columns=level_10,
)

message: [36000.228012451, 4.0, 100.0, 273400.0, 1.0]


,AskPrice10,AskSize10,BidPrice10,BidSize10
our book,274400,1400,-9999999999,0
LOBSTER,274400,1400,272400,800


Rebuilding the book and comparing it with LOBSTER.: 

181 181 of 181 244 rows differ. We can find the first mismatch is at row 64. We can see in the reconstructed book has no bid at level 10. LOBSTER shows 800 shares at 2 724 ticks. 

The first 63 rows match exactly, which shows that the translation of the messages is correct. The data only show 10 price levels, so some orders are missing. This makes it difficult to reconstruct the order book exactly.

## ii: Computing OFI and the queue imbalance I¹

In [7]:
#ii: OFI_t,w and I^1
from unito26.lob.lobster_session import LobsterMarketSession
from unito26.lob.messages import GridDepth, SweepSize
from unito26.lob.statistics import SessionStatistics

WINDOWS = (1, 5, 10, 30, 60)   # seconds
spec = SessionStatistics((GridDepth(1),), (SweepSize(100),), WINDOWS)
session = LobsterMarketSession.from_files(files, spec, GRID, WINDOW).coarsened(True)

stats = session.stats
book = session.lobster_book
print("session rows:", stats.shape[0], "| statistics:", stats.shape[1])
stats[["MidPrice", "OrderFlowContribution", "OrderFlowImbalance10", "QueueImbalance1"]].head()

session rows: 175209 | statistics: 38


,MidPrice,OrderFlowContribution,OrderFlowImbalance10,QueueImbalance1
TimeStamp,,,,
36000.031697,2734.5,NaN,NaN,-0.130644
36000.031784,2734.5,1400.0,NaN,0.056401
36000.032643,2734.5,1500.0,NaN,0.233171
36000.033928,2734.5,0.0,NaN,0.233171
36000.045602,2734.5,0.0,NaN,0.233171


In [8]:
#Checking the package against the definitions
bid_price, bid_size = book["BidPrice1"].to_numpy(float), book["BidSize1"].to_numpy(float)
ask_price, ask_size = book["AskPrice1"].to_numpy(float), book["AskSize1"].to_numpy(float)

# e_n by (1.4.1): row n against row n - 1
e_by_hand = np.full(len(book), np.nan)
e_by_hand[1:] = (
    (bid_price[1:] >= bid_price[:-1]) * bid_size[1:]
    - (bid_price[1:] <= bid_price[:-1]) * bid_size[:-1]
    - (ask_price[1:] <= ask_price[:-1]) * ask_size[1:]
    + (ask_price[1:] >= ask_price[:-1]) * ask_size[:-1]
)
# I^1 by (1.1.2)
i1_by_hand = (bid_size - ask_size) / (bid_size + ask_size)

print("max |e_n by hand - package|:", np.nanmax(np.abs(e_by_hand - stats["OrderFlowContribution"])))
print("max |I1  by hand - package|:", np.nanmax(np.abs(i1_by_hand - stats["QueueImbalance1"])))

max |e_n by hand - package|: 0.0
max |I1  by hand - package|: 0.0


In [9]:
#Missing Values
missing = pd.DataFrame({
    "undefined rows": [stats[f"OrderFlowImbalance{w}"].isna().sum() for w in WINDOWS],
    "rows within w seconds of row 0": [(stats.index < stats.index[0] + w).sum() for w in WINDOWS],
}, index=[f"OFI, w = {w}s" for w in WINDOWS])
missing.loc["e_n"] = [stats["OrderFlowContribution"].isna().sum(), 1]
missing.loc["I^1 (not covered)"] = [(stats["QueueImbalance1Covered"] == 0).sum(), 0]
missing

,undefined rows,rows within w seconds of row 0
"OFI, w = 1s",703,703
"OFI, w = 5s",1819,1819
"OFI, w = 10s",2879,2879
"OFI, w = 30s",5104,5104
"OFI, w = 60s",6014,6014
e_n,1,1
I^1 (not covered),0,0


In [10]:
columns = ["MidPrice", "OrderFlowContribution"] + [f"OrderFlowImbalance{w}" for w in WINDOWS] + ["QueueImbalance1"]
stats[columns].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
MidPrice,175209.0,2727.757,9.075,2712.500,2720.000,2727.000,2734.500,2748.5
OrderFlowContribution,175208.0,-1.246,783.776,-71702.000,-100.000,0.000,100.000,71702.0
OrderFlowImbalance1,174506.0,-1166.823,22354.597,-181900.000,-9221.000,0.000,7437.000,125354.0
OrderFlowImbalance5,173390.0,-1704.013,29641.952,-167855.000,-18253.000,272.000,14334.000,128548.0
OrderFlowImbalance10,172330.0,-3927.534,36543.484,-252868.000,-24416.000,-156.000,16885.000,128548.0
OrderFlowImbalance30,170105.0,-5610.789,52324.036,-265017.000,-34828.000,-1701.000,26145.000,147851.0
OrderFlowImbalance60,169195.0,-6972.154,69030.253,-292781.000,-52686.000,-7819.000,34424.000,217744.0
QueueImbalance1,175209.0,0.054,0.389,-0.999,-0.213,0.068,0.338,1.0


Computing OFI and the queue imbalance I¹

OFI was computed for 1, 5, 10, 30 and 60 seconds and $I^1$ for the best bid and ask sizes. Both match definitions (maximum difference: 0). Initial OFI values are undefined until each window is covered.

The mean OFI is negative for every window (- 1166.8 to - 6972.2 shares), which means more selling pressure than buying pressure over the morning. The mean of $I^1$ is slightly positive (0.054): the bid queue was on average a little larger than the ask queue. $I^1$ is a state at one moment, while OFI is a flow over a window, so the two do not have to point the same way.


## iii: How much does OFI move the price?

In [11]:
#iii regressing 
from unito26.lob import estimation as est
from unito26.lob import imbalance_regression as ir
from unito26.lob.assignment import implied_depth
from unito26.lob.messages import Window

mid = ir.aligned_mid_price(session)       # P^m in ticks, on the session's clock
flow = ir.aligned_order_flow(session)     # e_n

behind = [f"{side}Size{level}" for side in ("Bid", "Ask") for level in range(2, DEPTH + 1)]
print(f"mean size at the touch           : {np.nanmean((bid_size + ask_size) / 2):,.0f} shares")
print(f"mean size behind the touch (2-10): {book[behind].to_numpy(float).mean():,.0f} shares")

mean size at the touch           : 12,249 shares
mean size behind the touch (2-10): 13,291 shares


In [12]:
rows = []
for w in WINDOWS:
    ofi = ir.backward_sum(flow, Window(float(w)))
    change = ir.backward_change(mid, Window(float(w)))
    ok = np.isfinite(ofi) & np.isfinite(change)

    fit = est.ols(np.column_stack([np.ones(ok.sum()), ofi[ok]]), change[ok])
    slope = fit.coefficients[1]
    depth = np.nanmean(stats[f"AverageDepth{w}"])

    rows.append({
        "w (s)": w,
        "rows": ok.sum(),
        "slope": slope,
        "tau/(2 S)": 1 / (2 * depth),
        "slope / theory": slope * 2 * depth,
        "S at the touch": depth,
        "implied depth": implied_depth(slope),
        "R^2": fit.r_squared,
    })

regression = pd.DataFrame(rows).set_index("w (s)")
regression.style.format({"slope": "{:.3e}", "tau/(2 S)": "{:.3e}", "slope / theory": "{:.2f}",
                         "S at the touch": "{:,.0f}", "implied depth": "{:,.0f}", "R^2": "{:.3f}"})

,rows,slope,tau/(2 S),slope / theory,S at the touch,implied depth,R^2
w (s),,,,,,,
1,174506,2.493e-05,4.053e-05,0.62,"12,338","20,058",0.753
5,173390,2.478e-05,4.063e-05,0.61,"12,307","20,175",0.814
10,172330,2.384e-05,4.051e-05,0.59,"12,343","20,975",0.805
30,170105,2.382e-05,4.044e-05,0.59,"12,363","20,990",0.840
60,169195,2.492e-05,4.054e-05,0.61,"12,333","20,062",0.869


In [13]:
from unito26.lob.assignment import touch_residual

rows = []
for w in WINDOWS:
    ofi = ir.backward_sum(flow, Window(float(w)))
    change = ir.backward_change(mid, Window(float(w)))
    depth = regression.loc[w, "S at the touch"]
    eps = touch_residual(mid.times, bid_size, ask_size, float(w), depth)
    ok = np.isfinite(ofi) & np.isfinite(change) & np.isfinite(eps)

    gap = change[ok] - ofi[ok] / (2 * depth)              # what (1.4.3) leaves unexplained
    design = np.column_stack([np.ones(ok.sum()), ofi[ok]])
    slope = est.ols(design, change[ok]).coefficients[1]
    slope_without_eps = est.ols(design, change[ok] - eps[ok]).coefficients[1]
    theory = 1 / (2 * depth)

    rows.append({
        "w (s)": w,
        "share of Var(gap) explained by eps": 1 - np.var(gap - eps[ok]) / np.var(gap),
        "slope": slope,
        "slope without eps": slope_without_eps,
        "tau/(2 S)": theory,
        "share of slope gap closed": (slope_without_eps - slope) / (theory - slope),
        "|eps| <= tau, share of rows": np.mean(np.abs(eps[ok]) <= 1),
    })

residual = pd.DataFrame(rows).set_index("w (s)")
residual.style.format({"slope": "{:.3e}", "slope without eps": "{:.3e}", "tau/(2 S)": "{:.3e}",
                       "share of Var(gap) explained by eps": "{:.2f}", "share of slope gap closed": "{:.2f}",
                       "|eps| <= tau, share of rows": "{:.3f}"})

,share of Var(gap) explained by eps,slope,slope without eps,tau/(2 S),share of slope gap closed,"|eps| <= tau, share of rows"
w (s),,,,,,
1,0.45,2.493e-05,3.049e-05,4.053e-05,0.36,0.980
5,0.41,2.478e-05,2.941e-05,4.063e-05,0.29,0.977
10,0.36,2.384e-05,2.810e-05,4.051e-05,0.26,0.969
30,0.33,2.382e-05,2.774e-05,4.044e-05,0.24,0.953
60,0.23,2.492e-05,2.760e-05,4.054e-05,0.17,0.938


The OFI slope is about 60% of the theoretical value. The implied depth is around 20,000 shares, compared with 12,300 at the touch. The regressions have high R² values (0.75–0.87).

The residual ε explains 23–45% of the variation in the gap: 45% at $w = 1$ and 23% at $w = 60$. Removing ε brings the slope closer to the theoretical value (it closes 17–36% of the distance), but does not fully explain the difference.

The theoretical model assumes that orders only affect the best bid or ask, cannot reach beyond that quite, and that queues behind the best prices have the same fixed size. It also assumes that a new limit order improves the best price by one tick once the current best queue reaches that size. Real orders can instead arrive or be cancelled deeper in the book, and queue sizes vary. This may help explain why the market behaves as if it were deeper than the book at touch.

The residual matters most over short windows: when the best quotes do not change, some order flow changes the queue sizes without moving the mid-price. A high $R^2$ means that OFI explains much of the price movement over the same window, not that it forecasts the next move.

## iv: Does OFI forecast the next price move?

In [14]:
#iv
from unito26.lob.assignment import bucket_pairs

HORIZONS = (1, 5, 10, 30, 60)   # seconds

def forecast_grid(predictor_for_window, windows):
    # correlation of a predictor with the next bucket's mid-price move, for every (w, h)
    rows = []
    for w in windows:
        predictor = predictor_for_window(w)
        for h in HORIZONS:
            x, move = bucket_pairs(mid.times, predictor, mid, float(h))
            corr = np.corrcoef(x, move)[0, 1]
            rows.append({"w": w, "h": h, "n": len(x), "corr": corr, "t": corr * np.sqrt(len(x))})
    return pd.DataFrame(rows)

ofi_grid = forecast_grid(lambda w: ir.backward_sum(flow, Window(float(w))), WINDOWS)
ofi_grid.pivot(index="w", columns="h", values="corr").round(3)

h,1,5,10,30,60
w,,,,,
1,0.032,0.078,0.094,0.072,0.022
5,0.052,0.078,0.050,0.007,0.099
10,0.035,0.060,0.048,-0.045,-0.031
30,0.032,0.040,0.033,0.054,-0.018
60,0.035,0.049,0.051,0.062,-0.032


In [15]:
# t = corr * sqrt(n); |t| > 2 is "readable"
ofi_t = ofi_grid.pivot(index="w", columns="h", values="t")
ofi_t.round(1)

h,1,5,10,30,60
w,,,,,
1,2.5,3.0,2.5,1.1,0.2
5,4.1,3.0,1.3,0.1,1.1
10,2.7,2.3,1.3,-0.7,-0.3
30,2.5,1.5,0.9,0.8,-0.2
60,2.7,1.9,1.4,1.0,-0.4


OFI has weak correlations with future price movements (−0.045 to 0.099). The relationship is mainly readable at short horizons of 1–10 seconds, where some t-values exceed 2. The highest correlation (0.099) is not readable (t = 1.1).

Where the relationship can be read, the sign is positive: buying pressure (positive OFI) is followed by a rising mid-price, and selling pressure by a falling one. This is consistent with order flow tending to continue briefly in the same direction, rather than immediately reversing.

## v: Does $I^1$ forecast better than OFI?

In [16]:
#v
queue_imbalance = np.where(stats["QueueImbalance1Covered"].to_numpy(float) > 0,
                           stats["QueueImbalance1"].to_numpy(float), np.nan)

i1_grid = forecast_grid(lambda w: queue_imbalance, windows=[0])

best_ofi = ofi_grid.loc[ofi_grid.groupby("h")["t"].idxmax()].set_index("h")
comparison = pd.DataFrame({
    "n": i1_grid.set_index("h")["n"],
    "I^1: corr": i1_grid.set_index("h")["corr"],
    "I^1: t": i1_grid.set_index("h")["t"],
    "best OFI: w": best_ofi["w"],
    "best OFI: corr": best_ofi["corr"],
    "best OFI: t": best_ofi["t"],
})
comparison.round(3)

,n,I^1: corr,I^1: t,best OFI: w,best OFI: corr,best OFI: t
h,,,,,,
1,6076,0.179,13.983,5,0.052,4.051
5,1437,0.262,9.917,1,0.078,2.969
10,719,0.259,6.935,1,0.094,2.519
30,239,0.196,3.029,1,0.072,1.108
60,119,0.182,1.981,5,0.099,1.080


$I^1$ is more strongly correlated with the next price move than OFI at every tested horizon. Its correlations are 0.179–0.262, compared with OFI’s best 0.052–0.099. $I^1$ is clearly readable for $h=1$–30s ($t>2$), but not at 60s ($t=1.981$).
Remark 1.5.1 does not apply to the recorded data because real market participants may trade on information. This allows a fair comparison between OFI and I¹.


Part of this may still be mechanical: a thin queue is one market order from clearing, and clearing moves the mid. The data cannot separate this from information, but the result is consistent with the queue sizes carrying information about the next price move (adverse selection).


# Conclusion: 

This assignment shows how real market data can be used to understand order book movements and price changes (INTC, 21 June 2012, 10:00–12:00).

First, reconstructing the order book showed that limited data can lead to differences from the original book: 181 181 of 181 244 rows differ, starting at row 64 on level 10, because the data only show 10 price levels. The results also show that OFI helps explain current price changes ($R^2$ = 0.75–0.87, with a slope of about 60% of the theoretical value), but is less useful for predicting future movements (correlations of 0.03–0.09, readable only at horizons of 1–10 seconds).

In comparison, $I^1$ has a stronger relationship with future price movements (correlations of 0.18–0.26, about 2 to 3.5 times OFI's best), especially over short time periods (readable up to 30 seconds).

Overall, the main finding is that order flow helps explain price movements, while order book imbalance is more useful for predicting the next price movement. However, neither measure can perfectly predict future prices. The results are based on one stock over two hours of one day, so they should be checked on other stocks and days.